# Notebook 08: Integrate DepMap + IMPC

Add cancer dependency (DepMap) and embryonic lethality (IMPC) annotations
to the merged dataset.

**DepMap:** Common Essential, Strongly Selective, dependent cell line counts
**IMPC:** Embryonic lethal mouse knockouts linked to human orthologs via MGI

**Expected distributions from reference:**
- DepMap: not flagged=8,727; selective=2,557; essential=1,095; both=165
- IMPC: 908 lethal orthologs (422 Early, 356 Late, 130 Intermediate)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
REFERENCE_DIR = Path("../reference")

In [2]:
merged = pd.read_csv(PROCESSED_DIR / "07_hub_cage_with_cap_index.csv")
print(f"Loaded {len(merged):,} genes")

# Get the gene name column
gene_name_col = None
for c in merged.columns:
    if 'associated gene name' in c.lower() or c == 'Associated Gene Name':
        gene_name_col = c
        break
print(f"Gene name column: {gene_name_col}")

Loaded 12,811 genes
Gene name column: Associated Gene Name


## 1. DepMap Integration

Parse DepMap gene lists. Gene format: `GENE_NAME (ENTREZ_ID)`.

In [3]:
def parse_depmap_gene_list(filepath):
    """Parse DepMap Achilles gene list. One gene per row: GENE (ENTREZ_ID)."""
    df = pd.read_csv(filepath)
    col = df.columns[0]
    genes = df[col].dropna().astype(str)
    symbols = genes.str.extract(r'^([^(\s]+)', expand=False).str.strip()
    return set(symbols.dropna())


# Parse Common Essential gene list (Achilles controls file)
essential_path = RAW_DIR / "depmap_common_essential.csv"
if essential_path.exists():
    essential_genes = parse_depmap_gene_list(essential_path)
    print(f"Common Essential genes (Achilles): {len(essential_genes):,}")
else:
    essential_genes = set()
    print("WARNING: Common Essential file not found")

# Parse gene dependency summary
dep_summary_path = RAW_DIR / "depmap_gene_dep_summary.csv"
selective_genes = set()
dep_counts = {}
summary_essential = set()

if dep_summary_path.exists():
    dep_summary = pd.read_csv(dep_summary_path)
    print(f"\nGene dependency summary: {len(dep_summary):,} rows")
    print(f"Datasets: {dep_summary['Dataset'].unique().tolist()}")
    
    # Use Chronos_Combined (CRISPR-based), not RNAi_merged
    chronos = dep_summary[dep_summary["Dataset"].str.contains("Chronos", case=False, na=False)].copy()
    print(f"Chronos_Combined rows: {len(chronos):,}")
    
    # Strongly Selective genes
    ss_mask = chronos["Strongly Selective"] == True
    selective_genes = set(chronos.loc[ss_mask, "Gene"].astype(str).str.strip())
    print(f"Strongly Selective genes: {len(selective_genes):,}")
    
    # Common Essential from summary
    ce_mask = chronos["Common Essential"] == True
    summary_essential = set(chronos.loc[ce_mask, "Gene"].astype(str).str.strip())
    print(f"Common Essential genes (Chronos): {len(summary_essential):,}")
    
    # Dependent cell line counts
    for _, row in chronos.iterrows():
        gene = str(row["Gene"]).strip()
        n_dep = row["Dependent Cell Lines"]
        if pd.notna(n_dep):
            dep_counts[gene] = int(n_dep)
    print(f"Dependency counts: {len(dep_counts):,} genes")
    
    # Use Achilles list for Common Essential (matches Jason's approach)
    # but fall back to Chronos summary if Achilles list is smaller
    print(f"\nAchilles CE: {len(essential_genes)}, Chronos CE: {len(summary_essential)}")
else:
    print("WARNING: Gene dependency summary not found")

print(f"\nFinal: {len(essential_genes):,} essential, {len(selective_genes):,} selective, {len(dep_counts):,} dep counts")

Common Essential genes (Achilles): 1,247

Gene dependency summary: 35,330 rows
Datasets: ['DependencyEnum.RNAi_merged', 'DependencyEnum.Chronos_Combined']
Chronos_Combined rows: 18,494
Strongly Selective genes: 3,736
Common Essential genes (Chronos): 1,526


Dependency counts: 18,494 genes

Achilles CE: 1247, Chronos CE: 1526

Final: 1,247 essential, 3,736 selective, 18,494 dep counts


In [4]:
# If dep_counts is empty (gene_dep_summary didn't have it), fall back to CRISPRGeneDependency.csv
if not dep_counts:
    dep_path = RAW_DIR / "depmap_gene_dependency.csv"
    if dep_path.exists():
        print("Loading CRISPRGeneDependency.csv as fallback for cell line counts...")
        dep = pd.read_csv(dep_path, index_col=0)
        print(f"Loaded: {dep.shape[0]} cell lines x {dep.shape[1]} genes")
        
        for col in dep.columns:
            gene_symbol = col.split('(')[0].strip() if '(' in col else col.strip()
            n_dependent = (dep[col] >= 0.5).sum()
            dep_counts[gene_symbol] = n_dependent
        
        print(f"Dependency counts computed for {len(dep_counts):,} genes")
    else:
        print("WARNING: No dependency data available. Cell line counts will be 0.")

print(f"\nFinal dep_counts: {len(dep_counts):,} genes")
print(f"Essential genes: {len(essential_genes):,}")
print(f"Selective genes: {len(selective_genes):,}")


Final dep_counts: 18,494 genes
Essential genes: 1,247
Selective genes: 3,736


In [5]:
# Add DepMap columns to merged dataset
gene_names = merged[gene_name_col].astype(str)

merged["DepMap_Common_Essential"] = gene_names.isin(essential_genes).astype(int)
merged["DepMap_Strongly_Selective"] = gene_names.isin(selective_genes).astype(int)

# Flag status
def get_flag_status(row):
    ce = row["DepMap_Common_Essential"]
    ss = row["DepMap_Strongly_Selective"]
    if ce and ss:
        return "both"
    elif ce:
        return "common essential"
    elif ss:
        return "strongly selective"
    else:
        return "not flagged"

merged["DepMap_Flag_Status"] = merged.apply(get_flag_status, axis=1)

# Dependent cell line counts
merged["DepMap_Dependent_Cell_Lines"] = gene_names.map(dep_counts).fillna(0).astype(int)

# Distribution
print("DepMap Flag Status distribution:")
print(merged["DepMap_Flag_Status"].value_counts())

print(f"\nExpected from reference:")
print(f"  not flagged: 8,727")
print(f"  strongly selective: 2,557")
print(f"  common essential: 1,095")
print(f"  both: 165")

DepMap Flag Status distribution:
DepMap_Flag_Status
not flagged           9029
strongly selective    2647
common essential       990
both                   145
Name: count, dtype: int64

Expected from reference:
  not flagged: 8,727
  strongly selective: 2,557
  common essential: 1,095
  both: 165


## 2. IMPC Integration

Link human genes to mouse orthologs with embryonic lethality phenotypes.

In [6]:
# Load MGI human-mouse ortholog mapping
mgi_path = RAW_DIR / "mgi_human_mouse_orthologs.rpt"

if mgi_path.exists():
    mgi = pd.read_csv(mgi_path, sep='\t')
    print(f"MGI ortholog table: {len(mgi):,} rows")
    print(f"Columns: {list(mgi.columns)}")
    mgi.head()
else:
    print("WARNING: MGI ortholog file not found")

MGI ortholog table: 46,522 rows
Columns: ['DB Class Key', 'Common Organism Name', 'NCBI Taxon ID', 'Symbol', 'EntrezGene ID', 'Mouse MGI ID', 'HGNC ID', 'OMIM Gene ID', 'Genetic Location', 'Genome Coordinates (mouse: GRCm39 human: GRCh38)', 'Nucleotide RefSeq IDs', 'Protein RefSeq IDs', 'SWISS_PROT IDs']


In [7]:
# Parse MGI ortholog table to build mouse -> human gene symbol mapping
mgi_path = RAW_DIR / "mgi_human_mouse_orthologs.rpt"
human_to_mouse = {}  # human_symbol -> [mouse_symbols]
mouse_to_human = {}  # mouse_symbol -> human_symbol

if mgi_path.exists():
    mgi = pd.read_csv(mgi_path, sep='\t')
    print(f"MGI ortholog table: {len(mgi):,} rows")
    
    # Split by species
    human_rows = mgi[mgi["Common Organism Name"] == "human"][["DB Class Key", "Symbol"]].copy()
    mouse_rows = mgi[mgi["Common Organism Name"] == "mouse, laboratory"][["DB Class Key", "Symbol"]].copy()
    
    # Join on DB Class Key to get ortholog pairs
    ortho_pairs = mouse_rows.merge(human_rows, on="DB Class Key", suffixes=("_mouse", "_human"))
    print(f"Mouse-human ortholog pairs: {len(ortho_pairs):,}")
    
    # Build mappings
    for _, row in ortho_pairs.iterrows():
        h = row["Symbol_human"]
        m = row["Symbol_mouse"]
        mouse_to_human[m] = h
        if h not in human_to_mouse:
            human_to_mouse[h] = []
        human_to_mouse[h].append(m)
    
    print(f"Human genes with mouse orthologs: {len(human_to_mouse):,}")
    print(f"Mouse genes with human orthologs: {len(mouse_to_human):,}")
else:
    print("WARNING: MGI ortholog file not found")

MGI ortholog table: 46,522 rows
Mouse-human ortholog pairs: 24,590


Human genes with mouse orthologs: 19,366
Mouse genes with human orthologs: 20,181


In [8]:
# Load IMPC lethality data and classify by embryonic stage
import gzip

impc_path = RAW_DIR / "impc_genotype_phenotype_all.csv.gz"
lethal_mouse_genes = {}  # mouse_gene -> lethality_class

if impc_path.exists():
    impc = pd.read_csv(impc_path)
    print(f"IMPC entries: {len(impc):,}")
    
    # Filter to lethality-related MP terms
    lethality_mask = impc["mp_term_name"].astype(str).str.contains(
        "lethality|prenatal lethality", case=False, na=False
    )
    lethal_entries = impc[lethality_mask].copy()
    print(f"Lethality entries: {len(lethal_entries):,}")
    print(f"Unique mouse genes: {lethal_entries['marker_symbol'].nunique()}")
    print(f"\nMP terms:")
    print(lethal_entries["mp_term_name"].value_counts())
    
    # Classify by embryonic stage using procedure_name and mp_term_name
    def classify_lethality(row):
        proc = str(row["procedure_name"])
        mp = str(row["mp_term_name"])
        if "E9.5" in proc or "prior to organogenesis" in mp:
            return "Early lethal"
        elif "E12.5" in proc or "prior to tooth bud" in mp:
            return "Intermediate lethal"
        elif "E14.5" in proc or "E15.5" in proc or "E18.5" in proc:
            return "Late lethal"
        elif "prenatal lethality" in mp:
            return "Late lethal"
        elif "preweaning lethality" in mp:
            return "Late lethal"  # preweaning = postnatal death before weaning
        else:
            return "Late lethal"
    
    lethal_entries["lethality_class"] = lethal_entries.apply(classify_lethality, axis=1)
    
    # Take most severe classification per gene (Early > Intermediate > Late)
    severity = {"Early lethal": 0, "Intermediate lethal": 1, "Late lethal": 2}
    lethal_entries["severity"] = lethal_entries["lethality_class"].map(severity)
    lethal_per_gene = (
        lethal_entries
        .sort_values("severity")
        .drop_duplicates(subset="marker_symbol", keep="first")
    )
    
    # Build mouse gene -> lethality class mapping
    for _, row in lethal_per_gene.iterrows():
        lethal_mouse_genes[row["marker_symbol"]] = row["lethality_class"]
    
    print(f"\nLethal mouse genes classified: {len(lethal_mouse_genes):,}")
    from collections import Counter
    print(Counter(lethal_mouse_genes.values()))
else:
    print("WARNING: IMPC data file not found")

IMPC entries: 67,350
Lethality entries: 6,454
Unique mouse genes: 3029

MP terms:
mp_term_name
preweaning lethality, complete penetrance             3937
preweaning lethality, incomplete penetrance           1426
embryonic lethality prior to organogenesis             576
embryonic lethality prior to tooth bud stage           364
prenatal lethality prior to heart atrial septation     122
prenatal lethality                                      29
Name: count, dtype: int64

Lethal mouse genes classified: 3,029
Counter({'Late lethal': 2254, 'Early lethal': 573, 'Intermediate lethal': 202})


In [9]:
# Map IMPC lethality to human genes via MGI orthologs
gene_names = merged[gene_name_col].astype(str)

impc_lethal = []
impc_class = []
impc_mouse = []
impc_human = []

for gname in gene_names:
    # Look up mouse orthologs for this human gene
    mouse_genes = human_to_mouse.get(gname, [])
    found_lethal = False
    
    for mg in mouse_genes:
        if mg in lethal_mouse_genes:
            impc_lethal.append(1)
            impc_class.append(lethal_mouse_genes[mg])
            impc_mouse.append(mg)
            impc_human.append(gname)
            found_lethal = True
            break
    
    if not found_lethal:
        impc_lethal.append(0)
        impc_class.append("")
        impc_mouse.append("")
        impc_human.append("")

merged["IMPC_Embryonic_Lethal_Ortholog"] = impc_lethal
merged["IMPC_Embryonic_Lethal_Class_Ortholog"] = impc_class
merged["IMPC_Mouse_Source_Genes"] = impc_mouse
merged["IMPC_Human_Ortholog_Symbol"] = impc_human

total_lethal = sum(impc_lethal)
print(f"IMPC Embryonic Lethality distribution:")
print(f"  Total lethal orthologs: {total_lethal:,}")
lethal_classes = pd.Series([c for c in impc_class if c])
if len(lethal_classes) > 0:
    print(lethal_classes.value_counts())

print(f"\nExpected from Jason's reference:")
print(f"  Total lethal: 908")
print(f"  Early lethal: 422")
print(f"  Late lethal: 356")
print(f"  Intermediate lethal: 130")

IMPC Embryonic Lethality distribution:
  Total lethal orthologs: 2,521
Late lethal            1851
Early lethal            494
Intermediate lethal     176
Name: count, dtype: int64

Expected from Jason's reference:
  Total lethal: 908
  Early lethal: 422
  Late lethal: 356
  Intermediate lethal: 130


## 3. Validation Against Jason's Reference

In [10]:
ref = pd.read_csv(REFERENCE_DIR / "Lethality_DepMap_12544_Hub_CAGE_MERGE_with_CapIndex.csv")

# Compare DepMap distributions
print("=== DepMap Comparison ===")
our_depmap = merged["DepMap_Flag_Status"].value_counts().sort_index()
ref_depmap = ref["DepMap_Flag_Status"].value_counts().sort_index()

comp = pd.DataFrame({"Ours": our_depmap, "Reference": ref_depmap}).fillna(0).astype(int)
comp["Match"] = comp["Ours"] == comp["Reference"]
print(comp)

# Compare IMPC distributions
print("\n=== IMPC Comparison ===")
our_impc = merged["IMPC_Embryonic_Lethal_Ortholog"].sum()
ref_impc = ref["IMPC_Embryonic_Lethal_Ortholog"].sum()
print(f"Lethal orthologs: ours={our_impc}, ref={ref_impc}")

our_classes = merged[merged["IMPC_Embryonic_Lethal_Ortholog"] == 1]["IMPC_Embryonic_Lethal_Class_Ortholog"].value_counts()
ref_classes = ref[ref["IMPC_Embryonic_Lethal_Ortholog"] == 1]["IMPC_Embryonic_Lethal_Class_Ortholog"].value_counts()
comp_impc = pd.DataFrame({"Ours": our_classes, "Reference": ref_classes}).fillna(0).astype(int)
comp_impc["Match"] = comp_impc["Ours"] == comp_impc["Reference"]
print(comp_impc)

# Gene-level comparison for DepMap
print("\n=== Gene-level DepMap Comparison ===")
val = merged[["ensembl_id", "DepMap_Common_Essential", "DepMap_Strongly_Selective"]].merge(
    ref[["ensembl_id", "DepMap_Common_Essential", "DepMap_Strongly_Selective"]],
    on="ensembl_id", suffixes=("_ours", "_ref")
)
ce_match = (val["DepMap_Common_Essential_ours"] == val["DepMap_Common_Essential_ref"]).sum()
ss_match = (val["DepMap_Strongly_Selective_ours"] == val["DepMap_Strongly_Selective_ref"]).sum()
print(f"Common Essential exact match: {ce_match:,} / {len(val):,}")
print(f"Strongly Selective exact match: {ss_match:,} / {len(val):,}")

=== DepMap Comparison ===
                    Ours  Reference  Match
DepMap_Flag_Status                        
both                 145        165  False
common essential     990       1095  False
not flagged         9029       8727  False
strongly selective  2647       2557  False

=== IMPC Comparison ===
Lethal orthologs: ours=2521, ref=908
                                      Ours  Reference  Match
IMPC_Embryonic_Lethal_Class_Ortholog                        
Early lethal                           494        422  False
Intermediate lethal                    176        130  False
Late lethal                           1851        356  False

=== Gene-level DepMap Comparison ===
Common Essential exact match: 11,601 / 12,290
Strongly Selective exact match: 12,242 / 12,290


## 4. Save Output

In [11]:
output_path = PROCESSED_DIR / "08_hub_cage_depmap_impc.csv"
merged.to_csv(output_path, index=False)
print(f"Saved {len(merged):,} genes with all annotations to {output_path}")
print(f"Total columns: {len(merged.columns)}")

Saved 12,811 genes with all annotations to ../data/processed/08_hub_cage_depmap_impc.csv
Total columns: 47
